# OCR Extraction Pipeline — Setup & Test

Full litmus test for the extraction pipeline.

**Before starting:** Upload your sample PDFs/TIFFs to `/ocr/` using
Jupyter's file upload button.

This notebook will:
1. Check GPU and shared models
2. Load Qwen2.5-VL-7B directly
3. Load a sample document and check digital vs scanned
4. Run extraction
5. Inspect JSON output and experiment with formats
6. Process all pages
7. Launch Streamlit app

## 1. Check GPU and shared models

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader

In [ ]:
from pathlib import Path
import os

os.environ["HF_HOME"] = "/models/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/models/.cache/huggingface"
os.environ["HF_HUB_OFFLINE"] = "1"

models_dir = Path("/models/.cache/huggingface")
if models_dir.exists():
    model_dirs = [d.name for d in models_dir.iterdir() if d.name.startswith("models--")]
    print(f"Shared models PVC mounted. {len(model_dirs)} model(s):")
    for m in sorted(model_dirs):
        print(f"  {m}")
    has_vlm = any("Qwen2.5-VL" in d for d in model_dirs)
    if has_vlm:
        print("\nQwen2.5-VL found.")
    else:
        print("\nWARNING: Qwen2.5-VL not found!")
        print("Run: python /models/provision_shared_models.py download Qwen/Qwen2.5-VL-7B-Instruct")
else:
    print("WARNING: /models/.cache/huggingface not found.")
    print("Is the shared-models data volume attached?")

## 2. Load the model

Load Qwen2.5-VL-7B directly with transformers. Takes ~1-2 min.

In [ ]:
import time
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_NAME = "Qwen/Qwen2.5-VL-72B-Instruct"

print(f"Loading {MODEL_NAME}...")
t0 = time.time()

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

elapsed = time.time() - t0
print(f"Model loaded in {elapsed:.1f}s")
print(f"Device: {model.device}")

In [ ]:
!pip install -q qwen-vl-utils

In [ ]:
from qwen_vl_utils import process_vision_info

def run_vlm(messages, max_tokens=4096):
    """Run inference on the loaded model."""
    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_input], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_tokens)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0]

def extract_text(text, prompt, max_tokens=4096):
    """Digital path: send extracted text to model."""
    full_prompt = f"{prompt}\n\n---\nDOCUMENT TEXT:\n---\n{text}"
    messages = [{"role": "user", "content": [{"type": "text", "text": full_prompt}]}]
    return run_vlm(messages, max_tokens)

def extract_image(image, prompt, max_tokens=4096):
    """Scanned path: send image to model."""
    import base64, io
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    messages = [{"role": "user", "content": [
        {"type": "image", "image": f"data:image/png;base64,{b64}"},
        {"type": "text", "text": prompt},
    ]}]
    return run_vlm(messages, max_tokens)

print("Helper functions ready.")

In [ ]:
# ── Extraction prompt (per-page) ──────────────────────────────────────
# Adapted from the full document-level prompt for page-by-page processing.
# The assembly function merges per-page results into the final JSONL.

EXTRACTION_PROMPT = """Role & Objective: You are an expert data extraction assistant specializing in university grant administration documents. Data quality is paramount: do not abbreviate, shorten, or use external assumptions to fill in missing values.

Task: Extract all data from this single document page into the JSON structure below. Do not analyze data.

{
  "confidence_percentage": <float 0-100>,
  "confidence_narrative": "<brief note on extraction quality and comprehensiveness>",
  "has_annotation": <boolean>,
  "has_watermark": <boolean>,
  "signature_lines": {
    "has_signature_line": <boolean>,
    "has_valid_signature": <boolean>
  },
  "document_tags": ["<high-level grant admin tags, e.g. IRB, IACUC, Biosafety>"],
  "one_sentence_summary": "<one sentence summary>",
  "document_details": {
    "application_id": "", "application_type": "", "title": "",
    "requested_amount": null, "completed_date": "", "sub_document_type": ""
  },
  "stakeholders": [
    {
      "context_snippet": "<3-5 words near the stakeholder info>",
      "stakeholder_role": "<Principal Investigator | Co-Investigator | Collaborator | Key Personnel | Grants Administrative Contact | Sponsor Contact | Authorized Organizational Representative | Unknown>",
      "full_name": "", "first_name": "", "last_name": "",
      "email": "", "phone": "", "institution": "", "department": "",
      "position_title": "", "highest_education": "",
      "raw_stakeholder_text": "<verbatim text block containing stakeholder info>"
    }
  ],
  "addresses": [
    {
      "context_snippet": "<3-5 words near the address>",
      "addressee": "", "care_of": null,
      "address_line1": "", "address_line2": "",
      "city": "", "state_province": "", "postal_code": "", "country": "",
      "stakeholder_type": "<Funding Agency | Grantee Institution | Subrecipient | Principal Investigator | Grants Administrative Contact | Unknown>",
      "raw_address_text": "<verbatim text block containing the full address>"
    }
  ],
  "tables": [
    {
      "table_classification": "<Literal_Grid | Key_Value_Form | Standard_Table>",
      "table_data": "<see classification rules below>"
    }
  ],
  "narrative_responses": [
    {
      "prompt_or_header": "<exact question, section header, or 'General Body Text'>",
      "verbatim_text": "<complete, unsummarized text with [cite: N] markers>"
    }
  ],
  "other_metadata": {}
}

PROCESSING RULES:
- NARRATIVE EXTRACTION (CRITICAL FOR RAG): Extract ALL body text, paragraphs, memos, and application answers VERBATIM to ensure 100% document coverage. If text is part of a Q&A form, include the question in prompt_or_header. For unstructured letter/memo body, use "General Body Text". Do NOT summarize, truncate, or condense.
- CITATIONS: Add [cite: N] numbered tags after each distinct statement in narrative text, incrementing N from 1.
- TABLE CLASSIFICATION:
  * Literal_Grid: irregular tables without clear headers — table_data is a 2D array of strings (list of rows).
  * Key_Value_Form: label-value pairs (e.g. form cover sheets) — table_data is a single JSON object {label: value}.
  * Standard_Table: clear column headers — table_data is an array of objects with column headers as keys.
- SIGNATURES: Do NOT read handwriting. Only note if a signature LINE exists and if a signature is DETECTED.
- STAKEHOLDERS: Use ONLY the allowed stakeholder_role values listed above. If context does not make the role explicitly clear, use "Unknown". Capture raw_stakeholder_text verbatim.
- ADDRESSES: Use ONLY the allowed stakeholder_type values listed above. If unclear, use "Unknown". Place "Care Of"/"Attention" lines ONLY in care_of. Capture raw_address_text verbatim.
- Preserve ALL dollar amounts, dates, reference numbers exactly as they appear.
- Missing fields: use null or "" as appropriate. Escape all strings.
- Output ONLY valid JSON. No markdown fences, no introductory text."""


# ── Filename metadata parser ─────────────────────────────────────────
def parse_filename(filename):
    """Parse structured filename into FileNameMetaData.

    Rules from spec:
    - Six data elements in strict order, OFTEN split by single underscore.
    - Two underscores (__) next to each other = that data element is "".
    - After AwardID: Field2, Field3, Field4, Field5, then DocumentType.
    """
    import re
    stem = Path(filename).stem
    ext = Path(filename).suffix.lstrip('.')

    awd_match = re.search(r'_AWD-', stem)
    if not awd_match:
        return {
            "Drawer": stem, "AwardID": "", "Field2": "", "Field3": "",
            "Field4": "", "Field5": "", "DocumentType": stem,
            "DocumentTypeShort": stem, "FileType": ext
        }

    drawer = stem[:awd_match.start()]
    rest = stem[awd_match.start() + 1:]  # drop leading _

    # Split on _ — empty strings from __ indicate empty fields
    parts = rest.split('_')
    award_id = parts[0] if parts else ""

    # After AwardID: next 4 positions are Field2-Field5, rest is DocumentType
    remaining = parts[1:]
    field2 = remaining[0] if len(remaining) > 0 else ""
    field3 = remaining[1] if len(remaining) > 1 else ""
    field4 = remaining[2] if len(remaining) > 2 else ""
    field5 = remaining[3] if len(remaining) > 3 else ""
    doc_type = '_'.join(remaining[4:]) if len(remaining) > 4 else ""

    # DocumentTypeShort: strip category prefix (e.g. "A_RSP_Award_" -> "Notice_of_Award")
    doc_type_short = doc_type
    for cat in ("_Award_", "_Budget_", "_Report_", "_Agreement_", "_Proposal_"):
        idx = doc_type.find(cat)
        if idx >= 0:
            doc_type_short = doc_type[idx + len(cat):]
            break

    return {
        "Drawer": drawer, "AwardID": award_id,
        "Field2": field2, "Field3": field3, "Field4": field4, "Field5": field5,
        "DocumentType": doc_type, "DocumentTypeShort": doc_type_short,
        "FileType": ext
    }


# ── Document-level assembly (produces JSONL-ready dict) ──────────────
def assemble_document_jsonl(filename, page_results, model_name):
    """Combine per-page VLM results into the final document-level JSON per spec."""
    from datetime import datetime, timezone

    file_meta = parse_filename(filename)

    all_tables, all_narratives = [], []
    all_stakeholders, all_addresses = [], []
    all_tags = set()
    summaries = []
    has_annotation = has_watermark = False
    sig_info = {"PageNumber": None, "HasSignatureLine": False, "HasValidSignature": False}
    confidence_scores, confidence_narratives = [], []
    doc_details = {}
    other_meta = {}

    for pr in page_results:
        pg = pr["page"]
        d = pr.get("extracted", {})

        # Tables — map per-page snake_case to PascalCase
        for t in d.get("tables", []):
            all_tables.append({
                "PageNumber": pg,
                "TableClassification": t.get("table_classification",
                                             t.get("classification", "Standard_Table")),
                "TableData": t.get("table_data", t.get("rows", []))
            })

        # Narratives
        for n in d.get("narrative_responses", []):
            all_narratives.append({
                "SectionOrPage": f"PAGE {pg}",
                "PromptOrHeader": n.get("prompt_or_header", ""),
                "VerbatimText": n.get("verbatim_text", "")
            })

        # Stakeholders — map to PascalCase with full fields
        for s in d.get("stakeholders", []):
            if not any(v for v in s.values() if v):
                continue
            all_stakeholders.append({
                "PageNumber": pg,
                "ContextSnippet": s.get("context_snippet", ""),
                "StakeholderRole": s.get("stakeholder_role", "Unknown"),
                "FullName": s.get("full_name", ""),
                "FirstName": s.get("first_name", ""),
                "LastName": s.get("last_name", ""),
                "Email": s.get("email", ""),
                "Phone": s.get("phone", ""),
                "Institution": s.get("institution", ""),
                "Department": s.get("department", ""),
                "PositionTitle": s.get("position_title", ""),
                "HighestEducation": s.get("highest_education", ""),
                "RawStakeholderText": s.get("raw_stakeholder_text", "")
            })

        # Addresses — map to PascalCase with full fields
        for a in d.get("addresses", []):
            if not any(v for v in a.values() if v):
                continue
            all_addresses.append({
                "PageNumber": pg,
                "ContextSnippet": a.get("context_snippet", ""),
                "Addressee": a.get("addressee", ""),
                "CareOf": a.get("care_of"),
                "AddressLine1": a.get("address_line1", ""),
                "AddressLine2": a.get("address_line2", ""),
                "City": a.get("city", ""),
                "StateProvince": a.get("state_province", ""),
                "PostalCode": a.get("postal_code", ""),
                "Country": a.get("country", ""),
                "StakeholderType": a.get("stakeholder_type", "Unknown"),
                "RawAddressText": a.get("raw_address_text", "")
            })

        all_tags.update(d.get("document_tags", []))
        if d.get("one_sentence_summary"):
            summaries.append(d["one_sentence_summary"])
        if d.get("has_annotation"): has_annotation = True
        if d.get("has_watermark"): has_watermark = True

        sig = d.get("signature_lines", {})
        if sig.get("has_signature_line"):
            sig_info = {"PageNumber": pg, "HasSignatureLine": True,
                        "HasValidSignature": sig.get("has_valid_signature", False)}

        if d.get("confidence_percentage") is not None:
            confidence_scores.append(d["confidence_percentage"])
        if d.get("confidence_narrative"):
            confidence_narratives.append(d["confidence_narrative"])

        if not doc_details and d.get("document_details"):
            doc_details = d["document_details"]

        if d.get("other_metadata"):
            other_meta.update(d["other_metadata"])

    avg_conf = round(sum(confidence_scores) / len(confidence_scores), 1) if confidence_scores else 0.0
    now = datetime.now().astimezone().isoformat(timespec="seconds")

    return {
        "Filename": filename,
        "PageCount": len(page_results),
        "ConfidencePercentage": avg_conf,
        "ConfidenceNarrative": " ".join(confidence_narratives),
        "LLMModelAndVersion": model_name,
        "CurrentDateAndTime": now,
        "HasAnnotation": has_annotation,
        "HasWatermark": has_watermark,
        "SignatureLines": sig_info,
        "DocumentTags": sorted(all_tags),
        "OneSentenceNarrativeSummary": summaries,
        "FileNameMetaData": file_meta,
        "DocumentDetails": {
            "ApplicationID": doc_details.get("application_id", ""),
            "ApplicationType": doc_details.get("application_type", ""),
            "Title": doc_details.get("title", ""),
            "RequestedAmount": doc_details.get("requested_amount"),
            "CompletedDate": doc_details.get("completed_date", ""),
            "SubDocumentType": doc_details.get("sub_document_type", "")
        },
        "Stakeholders": all_stakeholders,
        "AddressesCollection": all_addresses,
        "TablesCollection": all_tables,
        "NarrativeResponses": all_narratives,
        "OtherMetadata": other_meta
    }


print("Extraction prompt, filename parser, and assembly function ready.")

## 3. Load a sample document

Upload your sample PDFs/TIFFs to `/ocr/` using Jupyter's file upload button.

In [ ]:
ocr_dir = Path("/ocr")
files = [f for f in ocr_dir.iterdir() if f.is_file()]
print("Files in /ocr/:")
for f in sorted(files):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")
if not files:
    print("\nNo files found. Upload your sample docs to /ocr/ first.")

In [ ]:
DOC_PATH = Path("/ocr/" + f.name)

assert DOC_PATH.exists(), f"File not found: {DOC_PATH}"
print(f"Document: {DOC_PATH.name} ({DOC_PATH.stat().st_size / 1024:.0f} KB)")

## 4. Check digital vs scanned

In [ ]:
import fitz

doc = fitz.open(str(DOC_PATH))
print(f"Pages: {len(doc)}\n")

page_info = []
for i, page in enumerate(doc):
    text = page.get_text("text").strip()
    has_text = len(text) >= 50
    page_info.append({"page": i, "text": text, "has_text": has_text})
    status = "DIGITAL" if has_text else "SCANNED"
    print(f"Page {i+1}: {status} ({len(text)} chars)")
    if has_text:
        print(f"  Preview: {text[:150]}...")
    print()

digital = sum(1 for p in page_info if p["has_text"])
scanned = sum(1 for p in page_info if not p["has_text"])
print(f"Summary: {digital} digital, {scanned} scanned")
doc.close()

In [ ]:
from PIL import Image

PAGE_IDX = 0  # Change this to test different pages
info = page_info[PAGE_IDX]

t0 = time.time()

if info["has_text"]:
    print(f"Page {PAGE_IDX+1}: Using DIGITAL path\n")
    raw_result = extract_text(info["text"], EXTRACTION_PROMPT)
else:
    print(f"Page {PAGE_IDX+1}: Using SCANNED path\n")
    doc = fitz.open(str(DOC_PATH))
    mat = fitz.Matrix(2.0, 2.0)
    pix = doc[PAGE_IDX].get_pixmap(matrix=mat)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()
    print(f"Rendered: {img.width}x{img.height}")
    display(img.resize((img.width // 3, img.height // 3)))
    raw_result = extract_image(img, EXTRACTION_PROMPT)

elapsed = time.time() - t0
print(f"Extraction took {elapsed:.1f}s")

In [ ]:
import json

# Strip markdown code fences if present
cleaned = raw_result.strip()
if cleaned.startswith("```"):
    cleaned = cleaned.split("\n", 1)[1]
    cleaned = cleaned.rsplit("```", 1)[0]

try:
    parsed = json.loads(cleaned)
    print("Valid JSON from VLM\n")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Invalid JSON: {e}\n")
    print("Raw output:")
    print(raw_result)

In [ ]:
# Quick test: parse the current document filename
meta = parse_filename(DOC_PATH.name)
print("FileNameMetaData:")
for k, v in meta.items():
    print(f"  {k}: {v!r}")

In [ ]:
import json
from PIL import Image

results = []
doc = fitz.open(str(DOC_PATH))

for info in page_info:
    page_num = info["page"] + 1
    t0 = time.time()

    if info["has_text"]:
        raw = extract_text(info["text"], EXTRACTION_PROMPT)
        method = "text_extraction"
    else:
        mat = fitz.Matrix(2.0, 2.0)
        pix = doc[info["page"]].get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        raw = extract_image(img, EXTRACTION_PROMPT)
        method = "vlm_ocr"

    elapsed = time.time() - t0

    # Parse VLM JSON output
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1]
        cleaned = cleaned.rsplit("```", 1)[0]
    try:
        extracted = json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"  WARNING: Page {page_num} returned invalid JSON, storing raw text")
        extracted = {"raw_text": raw, "confidence_percentage": 0,
                     "confidence_narrative": "Failed to parse structured output"}

    results.append({"page": page_num, "method": method,
                     "elapsed_ms": round(elapsed * 1000, 1), "extracted": extracted})
    print(f"Page {page_num}: {method} ({elapsed:.1f}s)")

doc.close()
print(f"\nDone. {len(results)} pages processed.")

In [ ]:
output = assemble_document_jsonl(
    filename=DOC_PATH.name,
    page_results=results,
    model_name=MODEL_NAME
)

# Write as JSONL (one JSON object per line)
out_path = Path(f"/ocr/{DOC_PATH.stem}_extracted.jsonl")
out_path.write_text(json.dumps(output) + "\n")
print(f"Saved JSONL to {out_path}\n")

# Pretty-print for inspection
print(json.dumps(output, indent=2))

## 9. Launch Streamlit app

Starts the extraction server in **local model mode** (no vLLM needed) and the Streamlit UI.

The server loads Qwen2.5-VL directly with transformers:
- **Digital pages** → text extraction + local LLM parse
- **Scanned pages** → local VLM OCR

Access at: `https://<cluster-host>/<project>/ocr-setup/proxy/8501/`

In [ ]:
import subprocess

repo_dir = "/home/user/KohakuRAG_UI"

# Free notebook model from GPU before server loads its own
if 'model' in dir() and model is not None:
    del model
    del processor
    torch.cuda.empty_cache()
    print("Freed notebook model from GPU.")

# Start extraction server in local mode (loads model with transformers, no vLLM)
env = {**os.environ, "LLM_BASE_URL": "local", "HF_HUB_OFFLINE": "1"}
server_proc = subprocess.Popen(
    ["python", "ocr_app/scripts/ocr_server.py"],
    env=env, cwd=repo_dir,
)
print(f"Extraction server starting (PID {server_proc.pid})...")
print("  Mode: LOCAL (transformers, no vLLM)")
print(f"  Model: {MODEL_NAME}")
print("  Digital pages -> text extraction + local LLM parse")
print("  Scanned pages -> local VLM OCR")

# Wait for server to load model and be ready
import time as _time
for i in range(120):
    _time.sleep(2)
    try:
        import httpx
        resp = httpx.get("http://localhost:8090/health", timeout=2.0)
        if resp.status_code == 200:
            info = resp.json()
            print(f"\nServer ready! LLM: {info.get('llm_model', '?')}")
            break
    except Exception:
        if i % 15 == 14:
            print(f"  Still loading model... ({(i+1)*2}s)")
else:
    print("WARNING: Server did not become ready within 4 min")

# Start Streamlit UI
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "ocr_app/app.py",
     "--server.port=8501", "--server.address=0.0.0.0", "--server.headless=true"],
    env={**os.environ, "OCR_SERVICE_URL": "http://localhost:8090"},
    cwd=repo_dir,
)
print(f"Streamlit started (PID {streamlit_proc.pid})")
print("\nAccess the UI at port 8501")
print("To stop: server_proc.terminate(); streamlit_proc.terminate()")